# Prueba lista: no necesitas subir nada
Ejecuta las primeras cinco celdas en orden. Se genera un **video artificial**, se procesa y se muestra el resultado. Permite comprobar la ejecución, no la precisión en personas reales. Después hay una prueba opcional con imágenes públicas y, al final, la carga de tu propio video.


In [ ]:
# Instala OpenCV solo si todavía no está disponible en Colab.
try:
    import cv2
except ImportError:
    %pip install -q opencv-python-headless
    import cv2
import numpy as np
from pathlib import Path
from IPython.display import display, Video
print("OpenCV:", cv2.__version__)


## 1. Detección y alertas

Se buscan líneas claras en la mitad inferior de la imagen. Las líneas candidatas a la izquierda y a la derecha del centro de la cámara se prolongan hasta una altura de referencia. Se compara la posición de la cámara con el ancho entre ambas. Es una **estimación visual**, no una medición en metros; depende de que la cámara esté centrada y orientada hacia adelante.


In [ ]:
# Parámetros para ajustar con videos reales.
UMBRAL_BORDE = 0.30  # Fracción del semiancho: 0.30 = alerta si |desplazamiento| > 30 %.
Y_REFERENCIA = 0.90  # Altura donde se mide (90 % de la imagen, desde arriba).
Y_INICIO = 0.52      # Se ignora la parte superior de la imagen.
BRILLO_MINIMO = 175  # Bajar si las líneas son poco brillantes; subir si hay falsos positivos.

def analizar_frame(frame):
    """Devuelve imagen anotada, estado y desplazamiento relativo, o None si no es fiable."""
    imagen = frame.copy()
    h, w = frame.shape[:2]
    y_ref = int(h * Y_REFERENCIA)
    centro = w / 2.0

    # Máscara de zonas claras y bordes; se eliminan regiones lejanas de la pista.
    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gris = cv2.GaussianBlur(gris, (5, 5), 0)
    _, claras = cv2.threshold(gris, BRILLO_MINIMO, 255, cv2.THRESH_BINARY)
    bordes = cv2.Canny(claras, 50, 150)
    bordes[:int(h * Y_INICIO), :] = 0
    lineas = cv2.HoughLinesP(bordes, 1, np.pi / 180, threshold=25,
                              minLineLength=max(20, int(h * 0.06)),
                              maxLineGap=max(15, int(w * 0.04)))
    izquierda, derecha = [], []
    if lineas is not None:
        # Normaliza las formas (N, 1, 4) y (N, 4) de distintas versiones.
        for x1, y1, x2, y2 in np.asarray(lineas).reshape(-1, 4):
            dy = y2 - y1
            if abs(dy) < h * 0.035:
                continue  # Rechaza líneas casi horizontales.
            # Prolonga cada segmento a la altura de referencia.
            x_ref = x1 + (y_ref - y1) * (x2 - x1) / dy
            if not (0 <= x_ref < w):
                continue
            # Se toman los segmentos que se abren hacia los lados al acercarse.
            pendiente = (x2 - x1) / dy
            if x_ref < centro and pendiente < -0.08:
                izquierda.append((x_ref, x1, y1, x2, y2))
            elif x_ref > centro and pendiente > 0.08:
                derecha.append((x_ref, x1, y1, x2, y2))

    estado, desplazamiento = "PISTA NO DETECTADA", None
    color = (0, 165, 255)
    if izquierda and derecha:
        # La línea más cercana al centro de cada lado es la hipótesis del carril actual.
        izq = max(izquierda, key=lambda linea: linea[0])
        der = min(derecha, key=lambda linea: linea[0])
        ancho = der[0] - izq[0]
        if w * 0.20 <= ancho <= w * 0.98:
            centro_carril = (izq[0] + der[0]) / 2
            desplazamiento = (centro - centro_carril) / (ancho / 2)
            for linea in (izq, der):
                _, x1, y1, x2, y2 = linea
                cv2.line(imagen, (x1, y1), (x2, y2), (0, 255, 0), 4)
            cv2.line(imagen, (int(izq[0]), y_ref), (int(der[0]), y_ref), (255, 180, 0), 3)
            cv2.circle(imagen, (int(centro_carril), y_ref), 7, (255, 180, 0), -1)
            cv2.circle(imagen, (int(centro), y_ref), 7, (255, 255, 255), -1)
            if abs(desplazamiento) > UMBRAL_BORDE:
                lado = "DERECHO" if desplazamiento > 0 else "IZQUIERDO"
                estado, color = "ACERCANDOSE A BORDE " + lado, (0, 0, 255)
            else:
                estado, color = "CENTRADO", (0, 200, 0)

    cv2.rectangle(imagen, (0, 0), (w, min(h, 75)), (0, 0, 0), -1)
    escala = max(0.45, min(0.9, w / 900))
    cv2.putText(imagen, estado, (10, 27), cv2.FONT_HERSHEY_SIMPLEX, escala, color, 2)
    if desplazamiento is not None:
        texto = f"Desplazamiento relativo: {desplazamiento:+.0%}"
        cv2.putText(imagen, texto, (10, 55), cv2.FONT_HERSHEY_SIMPLEX, escala * 0.8, (255, 255, 255), 2)
    return imagen, estado, desplazamiento


In [ ]:
# Genera un video artificial y lo analiza. No necesitas grabar ni subir archivos.
from IPython.display import Video, display
import subprocess
writer = cv2.VideoWriter('demo_procesada.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 20, (640,480))
original = cv2.VideoWriter('demo_entrada.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 20, (640,480))
if not writer.isOpened() or not original.isOpened():
    raise RuntimeError('No se pudo crear el video de demostración.')
conteo = {}
try:
    for i in range(200):
        # Cuatro etapas: centro, borde derecho, borde izquierdo, sin líneas.
        etapa = i // 50
        desplazamiento = [0, -80, 80, 0][etapa]
        frame = np.full((480,640,3), (45,55,75), dtype=np.uint8)
        if etapa != 3:
            for arriba, abajo in [(280,100),(360,540)]:
                cv2.line(frame, (arriba+desplazamiento,260),
                         (abajo+desplazamiento,479), (255,255,255), 8)
        original.write(frame)
        resultado, estado, _ = analizar_frame(frame)
        cv2.putText(resultado, 'SIMULACION - NO ES UNA PISTA REAL', (10,110),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
        writer.write(resultado)
        conteo[estado] = conteo.get(estado,0)+1
finally:
    writer.release()
    original.release()
subprocess.run(['ffmpeg','-y','-loglevel','error','-i','demo_procesada.mp4',
                '-c:v','libx264','-pix_fmt','yuv420p','demo_visible.mp4'],check=True)
print(conteo)
display(Video('demo_visible.mp4',embed=True))


## Opcional: imágenes reales
Fuente: https://github.com/vrai-group/TrackAid-DT . Consulta las condiciones de uso de los autores. Esta descarga depende del acceso a GitHub y de la estructura del repositorio; no fue verificada desde este entorno. Examina si las líneas detectadas coinciden con el carril.


In [ ]:
# OPCIONAL: busca hasta tres imágenes reales en el repositorio público TrackAid-DT.
# Solo descarga imágenes dentro de dataset; no ejecuta código del repositorio.
# Si no hay imágenes individuales accesibles, informa sin inventar muestras.
import requests
import random, json
from urllib.parse import quote
NUM_IMAGENES = 6
SEMILLA = 42  # Cambia este número para obtener otra muestra.
registro = []
from google.colab.patches import cv2_imshow
repo = 'vrai-group/TrackAid-DT'
r = requests.get(f'https://api.github.com/repos/{repo}/git/trees/main?recursive=1', timeout=30)
r.raise_for_status()
if r.json().get('truncated'):
    raise RuntimeError('GitHub devolvió una lista incompleta; no se seleccionará una muestra sesgada.')
rutas = [x['path'] for x in r.json().get('tree',[])
         if x['path'].startswith('dataset/')
         and x['path'].lower().endswith(('.jpg','.jpeg','.png'))
         and not any(k in x['path'].lower() for k in ['mask','label','annotation'])]
if not rutas:
    print('El repositorio no expone imágenes individuales en esa carpeta. La demo anterior sí funciona sin descargas.')
else:
    seleccion = random.Random(SEMILLA).sample(sorted(rutas), min(NUM_IMAGENES, len(rutas)))
    for ruta in seleccion:
        url = f'https://raw.githubusercontent.com/{repo}/main/{quote(ruta)}'
        respuesta = requests.get(url,timeout=30)
        respuesta.raise_for_status()
        foto = cv2.imdecode(np.frombuffer(respuesta.content,np.uint8),cv2.IMREAD_COLOR)
        if foto is None:
            print('Archivo no decodificable:',ruta)
            continue
        h,w = foto.shape[:2]
        foto = cv2.resize(foto,(640,max(1,int(h*640/w))))
        anotada,estado,_ = analizar_frame(foto)
        registro.append({'ruta': ruta, 'url': url, 'estado': estado, 'semilla': SEMILLA})
        print('Fuente:',url,'Estado:',estado)
        cv2_imshow(anotada)

with open("muestras_github.json", "w") as f:
    json.dump(registro, f, ensure_ascii=False, indent=2)
print("Registro guardado: muestras_github.json")


## 3. Sube y procesa tu video

Graba desde el centro del torso, apuntando hacia adelante y ligeramente hacia el suelo, con las dos líneas visibles. Empieza caminando con un acompañante en una zona controlada. Evita grabar a otras personas sin su consentimiento. El video se procesa en la sesión de Colab.


In [ ]:
from google.colab import files
subidos = files.upload()  # Selecciona un archivo de video corto.
if not subidos:
    raise ValueError("No se seleccionó ningún video.")
entrada = next(iter(subidos))
print("Video seleccionado:", entrada)


In [ ]:
# Procesa el video y guarda una copia con las alertas visuales.
# Limitar el tamaño hace más ligera la primera prueba.
MAX_ANCHO = 960
salida_temporal = "resultado_temporal.mp4"
salida = "resultado_carril.mp4"
cap = cv2.VideoCapture(entrada)
if not cap.isOpened():
    raise ValueError("No se pudo abrir el video. Prueba con un MP4.")
fps = cap.get(cv2.CAP_PROP_FPS)
if not np.isfinite(fps) or fps <= 0:
    fps = 25.0
w0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h0 = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if w0 <= 0 or h0 <= 0:
    cap.release()
    raise ValueError("No se pudieron leer las dimensiones del video.")
factor = min(1.0, MAX_ANCHO / w0)
w, h = max(2, int(w0 * factor) // 2 * 2), max(2, int(h0 * factor) // 2 * 2)
writer = cv2.VideoWriter(salida_temporal, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
if not writer.isOpened():
    cap.release()
    raise RuntimeError("No se pudo crear el video de salida.")
conteo = {}
try:
    while True:
        ok, cuadro = cap.read()
        if not ok:
            break
        cuadro = cv2.resize(cuadro, (w, h))
        anotado, estado, _ = analizar_frame(cuadro)
        conteo[estado] = conteo.get(estado, 0) + 1
        writer.write(anotado)
finally:
    cap.release()
    writer.release()
if not conteo:
    raise ValueError("El video no contiene cuadros legibles.")
print("Cuadros procesados por estado:", conteo)

# H.264 permite visualizar el resultado directamente en Colab.
import subprocess
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", salida_temporal,
                "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", salida], check=True)
display(Video(salida, embed=True))


In [ ]:
# Ejecuta esta celda para descargar el video procesado.
files.download(salida)


## 4. Cómo evaluar y mejorar

1. Revisa visualmente si las líneas verdes coinciden con los dos bordes de tu carril. Si no coinciden, la alerta no es confiable.
2. Prueba secuencias cortas centradas y acercándote a cada borde, siempre acompañado. Anota cuándo la alerta aparece, falla o detecta el carril equivocado.
3. Repite con sombras, curvas y movimiento al caminar; después prueba videos grabados trotando sin depender del sistema.
4. Ajusta `BRILLO_MINIMO`, `Y_INICIO` y `UMBRAL_BORDE` y vuelve a ejecutar desde la sección 1. El 30 % es un punto de partida arbitrario, no una distancia segura.
5. Siguiente fase: recopilar y etiquetar videos propios, medir errores y pérdidas de detección; si la visión clásica falla, evaluar segmentación entrenada con imágenes de pistas y agregar estabilización. El aviso sonoro o háptico y el tiempo real son etapas posteriores.

**Limitaciones:** las marcas de otros carriles, sombras, cruces, curvas y vibraciones pueden producir detecciones incorrectas. El sistema puede estar equivocado incluso cuando indica `CENTRADO`; si no ve ambos bordes debe mostrar `PISTA NO DETECTADA`. No sustituye a un guía.
